# FPL model results

Version 4 forecasts **expected Fantasy Premier League points for one
player-fixture** before the gameweek deadline. On the reserved 2025/26 season,
XGBoost reduced RMSE by **10.2%** versus recent-five across all 29,747 recorded
player-fixtures. This notebook reproduces the report tables from the public
aggregate summary.

In [1]:
from pathlib import Path
import json

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
SUMMARY_PATH = PROJECT_ROOT / "reports/public/model_results.json"
summary = json.loads(SUMMARY_PATH.read_text())
assert summary["schema_version"] == "fpl_public_model_results_v1"
pd.set_option("display.float_format", "{:.4f}".format)

## What the model could know

The target is recorded fixture points, including recorded non-appearances.
Older outcomes use the project's reviewed adjustments for defensive scoring
and verified corrections. Historical assist and bonus differences remain in
the source data. Match history, team context and availability use information
dated before the shared cutoff. Both fixtures in a double gameweek share
that history. Recent-five averages the latest five eligible outcomes and
uses a fitted position mean for cold starts.

[Scoring policy](../docs/fpl_rules.md) · [Data collection and preparation](../docs/fpl_collection_learning.md)

## Chronological evaluation

We screened development settings on expanding blocks in 2023/24 and used
2024/25 for development comparison. Both final forecasters were fitted
on 2022/23–2024/25 and held fixed throughout 2025/26. Eligible earlier
test-season results updated the histories as the season progressed.

The 2025/26 season appeared in descriptive EDA. It remained excluded from
predictive fitting and tuning and was scored only after the procedure was frozen.

Historical deadlines are approximated by the first fixture kickoff in
the gameweek minus 24 hours. A match enters history 24 hours after kickoff.
Archived availability precedes both the modelling cutoff and the recorded
FPL deadline. These rules approximate historical information timing.

In [2]:
display(pd.DataFrame(summary["periods"]).set_index("phase"))

,period,rows
phase,,
Development fit,2022/23–2023/24,56230
Development validation,2024/25,27283
Frozen refit,2022/23–2024/25,83513
Reserved evaluation,2025/26,29747


## Forecast accuracy

Development validation and the reserved evaluation report distinct fitting
periods and seasons. Both models use the full recorded player-fixture
population in each comparison.

In [3]:
overall_results = pd.DataFrame(summary["overall"])
display(overall_results.set_index(["phase", "model"]))
display(pd.Series(summary["metric_definitions"], name="Definition").to_frame())

season   rows   RMSE    MAE    bias
phase                  model                                            
Development validation Recent-five  2024/25  27283 2.1697 1.1211  0.0030
                       XGBoost v4   2024/25  27283 1.9770 1.0432  0.0058
Reserved evaluation    Recent-five  2025/26  29747 2.1182 1.0581  0.0139
                       XGBoost v4   2025/26  29747 1.9014 0.9604 -0.0144

,Definition
RMSE,"Square root of mean squared forecast error, in..."
MAE,"Mean absolute forecast error, in FPL points."
bias,Mean forecast minus observed points. Positive ...


## What the feature comparisons showed

Longer and weighted history brought a small development gain. Adding
pre-deadline availability and the playing chance published for the target
round improved the full-cohort forecasts. Grouped comparisons describe
sets of features and configurations. When a group stopped at the fold
screen its validation result is blank.

In [4]:
feature_results = pd.DataFrame(summary["feature_comparisons"]).rename(columns={
    "fold_rmse": "2023/24 fold RMSE",
    "validation_rmse": "2024/25 validation RMSE",
})
display(feature_results.set_index("features"))

,2023/24 fold RMSE,2024/25 validation RMSE
features,,
Fixed XGBoost reference,1.9915,2.0108
Remove team context,1.9994,NaN
Add longer and weighted history,1.9863,2.0062
"Add saves, defensive awards and bonus",1.9918,NaN
Weighted history with FPL status,1.9545,1.9799
Further add round-matched playing chance,1.9518,1.9770
Instead add league rest and congestion,1.9544,1.9795


Available regulars are players flagged available whose previous five
recorded fixtures averaged at least 60 minutes. In development, adding
availability and playing chance brought a smaller RMSE gain and a higher
MAE within that group. In the reserved comparison, XGBoost improved over
recent-five for available regulars. These results measure fixture-points
forecast accuracy. Squad value, captaincy and transfers require a separate
evaluation.

In [5]:
development_regulars = pd.DataFrame(summary["available_regulars"])
display(development_regulars.loc[
    development_regulars.phase.eq("Development validation"),
].set_index(["phase", "model"]))

cohort  ...    bias
phase                  model                                                     ...        
Development validation Weighted history  Available and prior-five mean >=60 min  ... -0.1535
                       XGBoost v4        Available and prior-five mean >=60 min  ... -0.0561

[2 rows x 5 columns]

## Reserved 2025/26 subgroup results

The tables compare the frozen forecasters on the same recorded player-fixtures.
Every row contributes to the overall comparison. Cold starts have zero eligible
prior records. Prior minutes are averaged over up to five available records.
Available regulars have FPL status available and a prior-minute average of at
least 60. That group's RMSE was 3.1160 for XGBoost and 3.4555 for recent-five.

Subgroups vary in size and outcome distribution. These tables describe
forecast error within each cohort.

In [6]:
def show_reserved_group(field, label):
    rows = pd.DataFrame(summary["reserved_subgroups"][field])
    counts = rows.groupby("group", sort=False).rows.first().rename("Rows")
    metrics = ["RMSE", "MAE", "bias"]
    paired = {"Cohort": counts.to_frame()}
    for model in ["Recent-five", "XGBoost v4"]:
        paired[model] = rows.loc[rows.model.eq(model)].set_index("group")[metrics]
    display(pd.concat(paired, axis=1).rename_axis(label))


show_reserved_group("position", "Position")
show_reserved_group("status", "FPL status")
show_reserved_group("status_coverage", "Availability coverage")

Cohort Recent-five               XGBoost v4               
           Rows        RMSE    MAE   bias       RMSE    MAE    bias
Position                                                           
DEF        9733      2.3282 1.2318 0.0117     2.0849 1.1110 -0.0683
FWD        3278      2.2506 1.1253 0.0036     2.0416 1.0576  0.0152
GK         3427      1.5653 0.6309 0.0156     1.4029 0.6111  0.0167
MID       13309      2.0459 1.0245 0.0176     1.8352 0.9163  0.0098

Cohort Recent-five                XGBoost v4               
              Rows        RMSE    MAE    bias       RMSE    MAE    bias
FPL status                                                             
Available    19610      2.4995 1.4265 -0.1001     2.2876 1.3628 -0.0307
Doubtful      1200      2.3240 1.4835  0.6912     1.9136 1.0646  0.0410
Injured       2053      1.1789 0.5003  0.4339     0.4411 0.1311  0.0274
Suspended      151      2.0585 1.4284  1.3648     0.3115 0.2106  0.1321
Unavailable   6427      0.1386 0.0161  0.0149     0.0527 0.0257 -0.0017
Ineligible     197      1.8683 1.2473  1.2270     0.4568 0.3098  0.2848
Unknown        109      1.2391 1.1759  1.0318     0.6657 0.3603  0.0425

Cohort Recent-five         ... XGBoost v4               
                        Rows        RMSE    MAE  ...       RMSE    MAE    bias
Availability coverage                            ...                          
Known                  29638      2.1208 1.0576  ...     1.9044 0.9626 -0.0146
Unknown                  109      1.2391 1.1759  ...     0.6657 0.3603  0.0425

[2 rows x 7 columns]

In [7]:
show_reserved_group("prior_minutes", "Prior minutes")
show_reserved_group("cold_start", "History coverage")
show_reserved_group("selection_cohort", "Pre-deadline cohort")

Cohort Recent-five                XGBoost v4               
                Rows        RMSE    MAE    bias       RMSE    MAE    bias
Prior minutes                                                            
cold start       844      2.3891 1.5884 -0.0183     2.2944 1.4713 -0.0062
0 to <30       19059      1.1542 0.3317 -0.1371     1.0699 0.3606 -0.0129
30 to <60       3484      2.7844 1.9567 -0.0027     2.4767 1.6720 -0.0261
60+             6360      3.4625 2.6720  0.4798     3.0692 2.3001 -0.0134

Cohort Recent-five                XGBoost v4               
                   Rows        RMSE    MAE    bias       RMSE    MAE    bias
History coverage                                                            
Cold start          844      2.3891 1.5884 -0.0183     2.2944 1.4713 -0.0062
Prior records     28903      2.1098 1.0426  0.0149     1.8887 0.9455 -0.0146

Cohort Recent-five         ... XGBoost v4               
                       Rows        RMSE    MAE  ...       RMSE    MAE    bias
Pre-deadline cohort                             ...                          
Available, prior 60+   5909      3.4555 2.6462  ...     3.1160 2.3632 -0.0360
Available, other      13701      1.9473 0.9005  ...     1.8174 0.9314 -0.0284
Other status          10137      1.0395 0.3453  ...     0.6964 0.1819  0.0173

[3 rows x 7 columns]

In [8]:
with pd.option_context("display.max_rows", 40):
    show_reserved_group("gameweek", "Gameweek")

Cohort Recent-five                XGBoost v4               
           Rows        RMSE    MAE    bias       RMSE    MAE    bias
Gameweek                                                            
1           690      2.5446 1.6700 -0.1763     2.4692 1.6418 -0.0285
2           705      2.8719 1.3176  0.1073     2.2328 1.1387 -0.0399
3           712      2.3505 1.2038  0.0222     1.9442 1.0406 -0.1024
4           740      2.1160 1.1317  0.0000     1.9828 1.0630 -0.1167
5           741      2.0437 1.0275  0.1942     1.7924 0.9215  0.0354
6           742      2.0080 1.0084  0.1523     1.8981 0.9218  0.0334
7           743      2.0256 1.0499  0.0104     1.8215 0.9158 -0.0544
8           745      2.2050 1.0790 -0.0357     2.0434 0.9892 -0.0714
9           746      2.1470 1.0342 -0.0215     1.9687 0.9612 -0.0449
10          747      1.9943 1.0326 -0.0078     1.8168 0.9408  0.0024
11          752      2.1374 1.0601 -0.0835     1.9775 1.0062 -0.0767
12          755      2.2219 1.0774  0.0215     2.0270 0.9826 -0.0324
13          755      2.2072 1.0804  0.0666     1.9606 0.9852  0.0463
14          758      2.3361 1.1305 -0.0636     2.1050 1.0194 -0.0839
15          759      2.0320 1.0260 -0.0071     1.9432 0.9922 -0.0395
16          760      2.1957 1.0861  0.0006     2.0478 1.0050 -0.0573
17          770      2.2527 1.1814  0.0204     1.9051 0.9433 -0.0954
18          775      2.2006 1.0821  0.0369     1.8447 0.8824 -0.0546
19          780      2.2023 1.1287 -0.0488     1.9448 0.9789 -0.1367
20          790      2.1981 1.0862  0.0389     1.8994 0.9184 -0.0366
21          795      1.9415 0.9775  0.0413     1.7489 0.8897 -0.0106
22          799      2.0124 1.0660  0.0183     1.7635 0.9368 -0.0332
23          803      2.1149 1.1146  0.0416     1.8486 0.9674  0.0178
24          811      2.1210 1.0736  0.0439     1.8345 0.9219  0.0278
25          817      2.0056 1.0037  0.0574     1.7896 0.8932  0.0315
26          896      2.0350 1.0049  0.0693     1.8326 0.8996  0.0566
27          818      1.9749 0.9384 -0.0258     1.8254 0.8897 -0.0005
28          819      1.9715 0.9450  0.0671     1.7873 0.9023  0.0748
29          820      1.9140 0.9348 -0.0391     1.8343 0.9075 -0.0054
30          822      1.9858 1.0081 -0.0994     1.7933 0.9428 -0.0465
31          664      2.0211 0.9701 -0.0751     1.8132 0.8908 -0.0213
32          826      2.1615 0.9808  0.0089     1.9936 0.9213  0.0132
33         1077      1.9359 0.9464 -0.0359     1.7715 0.8846 -0.0163
34          582      1.9548 1.0091  0.0559     1.7533 0.8930  0.0243
35          832      2.0672 1.0147  0.0748     1.8494 0.9219  0.0581
36          920      2.0282 0.9926  0.0370     1.8185 0.9199  0.0458
37          840      1.9339 0.9223  0.0366     1.7531 0.8774  0.0662
38          841      1.9907 1.0027  0.0196     1.8174 0.9437  0.0643

## From data to gameweek forecasts

| Stage | Output | Status |
|---|---|---|
| Dated FPL data and historical results | Roster, fixture calendar and eligible result history | Collected and checked |
| Cutoff-aware feature preparation | One predictor row per player-fixture | Used for development, reserved scoring and upcoming inference |
| Frozen model forecast | Expected points for each scheduled player-fixture | Forecasting milestone complete |
| Gameweek aggregation | Sum of fixture expectations with explicit coverage | Complete for the dated batch |
| Squad selection | Legal squad, substitutions and captaincy decisions | Separate evaluation milestone |

A verified blank contributes zero fixture points. A roster entry with
incomplete fixture or forecast coverage retains a missing total.
The dated GW12 export uses completed results through GW10. The export needs
refreshing after GW11 results become available, before the GW12 deadline.

[Collection guide](../docs/fpl_collection_learning.md)

In [9]:
batch = summary["reporting"]
forecast_batch = pd.DataFrame([{
    "season": batch["forecast_season"],
    "gameweek": batch["forecast_gameweek"],
    "captured UTC": batch["forecast_snapshot_utc"],
    "deadline UTC": batch["forecast_deadline_utc"],
    "history through GW": batch["history_through_gameweek"],
    "fixture forecasts": batch["fixture_rows"],
    "roster totals": batch["gameweek_roster_rows"],
}]).set_index(["season", "gameweek"])
with pd.option_context("display.max_columns", None, "display.width", 145):
    display(forecast_batch)

,,captured UTC,deadline UTC,history through GW,fixture forecasts,roster totals
season,gameweek,,,,,
2026/27,12,2026-11-21T16:45:21+00:00,2026-11-27T18:30:00+00:00,10,672,672
